# RL Run Analysis

This notebook analyzes W&B runs from RL training experiments (using `hf_trainer` with GRPOTrainer).
Fetches and visualizes reward metrics, parsing statistics, training progress, and sample outputs.

**Setup:** Ensure wandb is configured (`wandb login` or API key in `.env`).

In [ ]:
import html
import json
import typing

import IPython.display as ipy_display
import matplotlib.figure
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import pyine.utils.filesystem
import pyine.utils.notebooks as nb_utils
import pyine.utils.reprod
import pyine.utils.wandb_utils as wandb_utils

In [ ]:
pyine.utils.reprod.entrypoint_setup()
nb_utils.setup_notebook_plotting(use_seaborn=True, seaborn_style="whitegrid")

# ------------ CONFIGURATION ------------
WANDB_PROJECT = "pyine-tests"  # change to your project
WANDB_ENTITY = None  # optional: your team/user entity

# option 1: specify run directly by URL or ID (takes priority)
WANDB_RUN_URL = (
    "https://wandb.ai/lawzero-default/pyine-tests/runs/k28c1xw9"  # e.g., "https://wandb.ai/entity/project/runs/abc123"
)
WANDB_RUN_ID = ""  # e.g., "abc123"

# option 2: search for runs using filters (used when URL/ID not specified)
# W&B API filters (applied server-side) - use MongoDB-style query syntax
RUN_FILTERS: dict[str, typing.Any] = {
    # RL runs have num_generations > 0 (GRPO generates multiple completions per prompt)
    "config.num_generations": {"$gt": 0},
    # "state": "finished",  # uncomment to only show completed runs
    # "group": "...",  # uncomment to filter by run group
}
# optional client-side filter (set to None to skip); applied after server-side filters
RUN_FILTER_FN: typing.Callable[[typing.Any], bool] | None = None
SELECTED_RUN_IDX = 0  # which run to select from search results (0 = most recent)

# optional: parquet cache path for large runs (set to None to disable)
history_cache_dir = pyine.utils.filesystem.get_data_cache_subdir("notebooks", "cached_wandb_runs")

In [ ]:
# load run: try direct URL/ID first, then fall back to filter-based search
if WANDB_RUN_URL or WANDB_RUN_ID:
    # option 1: direct run specification
    run = wandb_utils.get_wandb_run(WANDB_RUN_URL, WANDB_RUN_ID, WANDB_PROJECT, WANDB_ENTITY)
    print(f"Loaded run directly: {run.name} ({run.url})")
else:
    # option 2: search for runs using filters
    print(f"Searching for runs with filters: {RUN_FILTERS}")
    runs = wandb_utils.fetch_runs(
        project=WANDB_PROJECT,
        entity=WANDB_ENTITY,
        filters=RUN_FILTERS if RUN_FILTERS else None,
        order="-created_at",  # newest first
        per_page=50,
    )
    # apply client-side filter if provided (e.g., to filter for RL runs)
    if RUN_FILTER_FN is not None:
        runs = [r for r in runs if RUN_FILTER_FN(r)]
        print(f"After client-side filter: {len(runs)} RL runs")
    if not runs:
        raise ValueError(f"No runs found matching filters (server: {RUN_FILTERS}, client: {RUN_FILTER_FN})")
    print(f"Found {len(runs)} matching runs:")
    for idx, r in enumerate(runs[:10]):  # show up to 10
        marker = " <-- selected" if idx == SELECTED_RUN_IDX else ""
        print(f"  [{idx}] {r.group}/{r.name} ({r.state}) - {r.created_at}{marker}")
    if len(runs) > 10:
        print(f"  ... and {len(runs) - 10} more")
    if len(runs) <= SELECTED_RUN_IDX:
        raise ValueError(f"SELECTED_RUN_IDX={SELECTED_RUN_IDX} but only {len(runs)} runs found")
    run = runs[SELECTED_RUN_IDX]
    print(f"\nSelected run: {run.name} ({run.url})")

summary = dict(run.summary)
config = dict(run.config).get("main_config", {})
print(f"State: {run.state}, Created: {run.created_at}")

In [ ]:
def get_nested_config_value(
    config: dict[str, typing.Any],
    dotted_key: str,
) -> typing.Any | None:
    """Retrieve a value from a nested config dict using dot notation (e.g., 'grpo_config.beta')."""
    parts = dotted_key.split(".")
    current = config
    for part in parts:
        if isinstance(current, dict) and part in current:
            current = current[part]
        else:
            return None
    return current


print(f"Run Configuration: {run.name} ({run.url})")
config_keys_of_interest = [
    "base_model",
    "datamodule_config.lmdb_paths",
    "grpo_config.per_device_train_batch_size",
    "grpo_config.per_device_eval_batch_size",
    "grpo_config.gradient_accumulation_steps",
    "grpo_config.learning_rate",
    "grpo_config.weight_decay",
    "grpo_config.gradient_checkpointing",
    "grpo_config.num_train_epochs",
    "grpo_config.num_generations",
    "grpo_config.max_completion_length",
    "grpo_config.max_prompt_length",
    "grpo_config.temperature",
    "grpo_config.beta",
]
for key in config_keys_of_interest:
    value = get_nested_config_value(config, key)
    if value is not None:
        if isinstance(value, (tuple, list)):
            print(f"  {key}:")
            for item in value:
                print(f"\t- {repr(item)}")
        elif isinstance(value, dict):
            for k, v in value.items():
                print(f"  {key}.{k}: {repr(v)}")
        else:
            print(f"  {key}: {repr(value)}")

In [ ]:
available_keys = wandb_utils.discover_metric_keys(run)

print("Available Metrics in This Run:")
for category, keys in available_keys.items():
    if keys:
        print(f"\n  {category} ({len(keys)} keys):")
        for key in keys:
            print(f"    - {key}")
    else:
        print(f"\n  {category}: (none)")

In [ ]:
print("\nRun Summary (key metrics):")
summary_keys_of_interest = [
    "eval/rewards/TRLRewardAdapter/mean",
    "eval/reward/run/total/mean",  # check: should be the same as eval/rewards/TRLRewardAdapter/mean
    "eval/reward/run/total/count",
    "eval/parsing/missing_answer_ratio",
    "eval/parsing/reasoning_length_tokens/mean",
    "eval/throughput/samples_per_second",
    "eval/failures/failure_count",
    "train/failures/failure_count",
    "train/reward/run/total/mean",
    "train/reward/run/total/count",
    "train/parsing/missing_answer_ratio",
    "train/parsing/reasoning_length_tokens/mean",
    "train/throughput/samples_per_second",
    "train/generation_count",
    "train/batch_count",
    "train/global_step",
    "train/epoch",
    "train/gpu/utilization_gpu_percent/mean",
    "train/gpu/vram_used_percent/mean",
]
for key in summary_keys_of_interest:
    if key in summary:
        print(f"  {key}: {summary[key]}")

In [ ]:
# fetch important metrics (focus mostly on run-wise logs, not sporadic logs)
important_key_parts = [
    "/reward/run/total/",
    "/reward/run/terms/",
    "/reward/batch/",
    "/reward/metrics/verbosity/factor/clip_ratio/",
    "/parsing/output_length_tokens/",
    "/parsing/reasoning_length_tokens/",
    "/parsing/answer_length_tokens/",
    "/clip_ratio/",
    "/throughput/",
    "/gpu/",
]
important_key_suffixes = [
    "/failures/failure_count",
    "/parsing/missing_reasoning_ratio",
    "/parsing/missing_answer_ratio",
    "/entropy",
    "/kl",
    "/loss",
    "/grad_norm",
    "/step_time",
    "/num_tokens",
    "/samples_per_second",
    "/steps_per_second",
]
available_keys_flat = [key for keys in available_keys.values() for key in keys]
important_key_exact = ["loss", "clip_ratio"]  # keys that may be logged without a split prefix
important_keys = sorted(
    set(
        [
            key
            for key in available_keys_flat
            if any(s in key for s in important_key_parts)
            or any(key.endswith(s) for s in important_key_suffixes)
            or key in important_key_exact
        ]
        + available_keys["step_keys"]  # always grab all step keys as important
    )
)

# only cache history for finished runs (incomplete runs would have stale cached data)
history_cache_path = history_cache_dir / f"{run.id}.parquet" if run.state != "running" else None
if history_cache_path is None:
    print(f"Run state is '{run.state}' - caching disabled (will fetch fresh data)")
history_df = wandb_utils.fetch_history_df(
    run,
    keys=important_keys if important_keys else None,
    cache_path=history_cache_path,
)
step_key = wandb_utils.resolve_step_key(history_df)
time_key = wandb_utils.resolve_time_key(history_df)

print(f"\nFetched {len(history_df)} history rows with {len(history_df.columns)} columns")
print(f"Using step key: {step_key}")
if time_key:
    print(f"Time key available: {time_key}")

In [ ]:
history_df  # noqa: B018 (for display purposes)

## Reward Metrics

In [ ]:
# color palette for multi-line plots (colorblind-friendly)
TERM_COLORS = ["#2C7BB6", "#D7191C", "#1A9641", "#FDAE61", "#9970AB", "#E7298A", "#66A61E", "#E6AB02"]
SPLIT_COLORS = {"train": TERM_COLORS[0], "eval": TERM_COLORS[1]}


def _derive_sibling_key(mean_key: str, suffix: str) -> str | None:
    """Derive a sibling key (e.g. std, count) from a /mean key."""
    if mean_key.endswith("/mean"):
        return mean_key[:-5] + "/" + suffix
    return None


def _compute_percentile_x_limits(
    all_x_values: list | np.ndarray,
    percentile_range: tuple[float, float],
    padding_fraction: float = 0.05,
) -> tuple[float, float] | None:
    """Compute x-axis limits from percentiles with padding."""
    if len(all_x_values) == 0:
        return None
    x_low = float(np.percentile(all_x_values, percentile_range[0]))
    x_high = float(np.percentile(all_x_values, percentile_range[1]))
    if x_low >= x_high:
        return None
    padding = (x_high - x_low) * padding_fraction
    return (x_low - padding, x_high + padding)


def _plot_metric_series(
    ax: plt.Axes,
    x_vals: np.ndarray,
    mean_vals: np.ndarray,
    color: str,
    label: str,
    std_vals: np.ndarray | None = None,
    counts: np.ndarray | None = None,
    show_ci: bool = True,
    show_scatter: bool = True,
) -> None:
    """Plot a single metric series (line + scatter + optional CI band) on an axis."""
    if show_ci and std_vals is not None:
        if counts is not None:
            safe_counts = np.maximum(counts, 1)
            se = std_vals / np.sqrt(safe_counts)
            lower = mean_vals - 1.96 * se
            upper = mean_vals + 1.96 * se
        else:
            lower = mean_vals - std_vals
            upper = mean_vals + std_vals
        ax.fill_between(x_vals, lower, upper, alpha=0.2, color=color)
    if show_scatter:
        ax.scatter(x_vals, mean_vals, alpha=0.4, s=12, color=color)
    ax.plot(x_vals, mean_vals, color=color, linewidth=1.5, alpha=0.8, label=label)


def _configure_metric_ax(
    ax: plt.Axes,
    step_key: str,
    title: str,
    x_limits: tuple[float, float] | None = None,
    show_legend: bool = True,
    legend_loc: str = "best",
) -> None:
    """Apply common axis configuration for reward metric plots."""
    ax.set_xlabel(step_key)
    ax.set_ylabel("reward")
    ax.set_title(title)
    if x_limits is not None:
        ax.set_xlim(*x_limits)
    ax.grid(axis="y", alpha=0.3)
    if show_legend:
        ax.legend(loc=legend_loc, fontsize=8)


def plot_reward_over_time(
    history_df: pd.DataFrame,
    mean_keys: list[str] | str = ("train/reward/run/total/mean", "eval/reward/run/total/mean"),
    step_key: str = "train/global_step",
    ax: plt.Axes | None = None,
    figsize: tuple[int, int] = (12, 5),
    title: str | None = None,
    show_scatter: bool = True,
    show_legend: bool = True,
    show_ci: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
    colors: list[str] | None = None,
) -> matplotlib.figure.Figure:
    """Plot reward metrics over training steps using logged mean/std values.

    Args:
        history_df: DataFrame with history data.
        mean_keys: Name(s) of the mean reward column(s) to plot. Can be a single string or list.
        step_key: Name of the step column (x-axis).
        ax: Optional matplotlib axes to plot on.
        figsize: Figure size if creating new figure.
        title: Chart title (auto-generated if None).
        show_scatter: Whether to show individual data points.
        show_legend: Whether to show the legend.
        show_ci: Whether to show error bars (using logged std values).
        x_percentile_range: Percentile range for x-axis limits, e.g. (1, 99).
        colors: Optional list of colors for each mean_key. Defaults to TERM_COLORS.

    Returns:
        The matplotlib Figure object.
    """
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()
    if isinstance(mean_keys, str):
        mean_keys = [mean_keys]
    else:
        mean_keys = list(mean_keys)
    valid_mean_keys = [k for k in mean_keys if k in history_df.columns]
    if not valid_mean_keys:
        ax.text(0.5, 0.5, f"None of {mean_keys} found in history", ha="center", va="center", transform=ax.transAxes)
        return fig
    if colors is None:
        colors = TERM_COLORS
    # compute x-axis limits
    all_x_values = []
    for mean_key in valid_mean_keys:
        valid_df = history_df[[step_key, mean_key]].dropna()
        if len(valid_df) > 0:
            all_x_values.extend(valid_df[step_key].values)
    x_limits = _compute_percentile_x_limits(all_x_values, x_percentile_range) if x_percentile_range else None
    # plot each key
    for key_idx, mean_key in enumerate(valid_mean_keys):
        color = colors[key_idx % len(colors)]
        std_key = _derive_sibling_key(mean_key, "std")
        count_key = _derive_sibling_key(mean_key, "count")
        has_std = std_key is not None and std_key in history_df.columns
        has_count = count_key is not None and count_key in history_df.columns
        cols = [step_key, mean_key]
        if has_std:
            cols.append(std_key)
        if has_count:
            cols.append(count_key)
        valid_df = history_df[cols].dropna(subset=[step_key, mean_key]).sort_values(step_key).reset_index(drop=True)
        if len(valid_df) == 0:
            continue
        label_prefix = "train" if "train/" in mean_key else ("eval" if "eval/" in mean_key else mean_key)
        _plot_metric_series(
            ax,
            x_vals=valid_df[step_key].values,
            mean_vals=valid_df[mean_key].values,
            color=color,
            label=f"{label_prefix} (n={len(valid_df)})",
            std_vals=valid_df[std_key].fillna(0).values if has_std else None,
            counts=valid_df[count_key].fillna(1).values if has_count else None,
            show_ci=show_ci,
            show_scatter=show_scatter,
        )
    _configure_metric_ax(
        ax, step_key, title or "Total Reward Over Training", x_limits, show_legend, legend_loc="upper left"
    )
    return fig


# side-by-side plots: run-wise (left) and batch-wise (right) reward metrics
fig, (ax_run, ax_batch) = plt.subplots(1, 2, figsize=(20, 5))
plot_reward_over_time(
    history_df,
    mean_keys=("train/reward/run/total/mean", "eval/reward/run/total/mean"),
    step_key=step_key,
    ax=ax_run,
    title="Run-wise Reward Over Training",
)
plot_reward_over_time(
    history_df,
    mean_keys="train/reward/batch/mean",
    step_key=step_key,
    ax=ax_batch,
    title="Batch-wise Reward Over Training",
)
plt.tight_layout()
plt.show()

In [ ]:
def _parse_term_summary_tables(
    train_table: pd.DataFrame | None,
    eval_table: pd.DataFrame | None,
    step_mapping: pd.Series | None = None,
    id_column: str = "term",
) -> dict[str, dict[str, pd.DataFrame]]:
    """Parse term_summary wandb tables into per-term, per-split DataFrames.

    Each summary table has columns [<id_column>, mean, std, min, max, count] with one
    row per item per logged step (_logged_step added by fetch_tables_with_steps).

    Args:
        train_table: Train term_summary table (or None).
        eval_table: Eval term_summary table (or None).
        step_mapping: Optional Series mapping wandb _step -> resolved step key values.

    Returns:
        Dict mapping term_name -> {"train": df, "eval": df} where each df has
        columns [step, mean, std, count].
    """
    result: dict[str, dict[str, pd.DataFrame]] = {}
    for split, table in [("train", train_table), ("eval", eval_table)]:
        if table is None or len(table) == 0:
            continue
        if "_logged_step" not in table.columns or id_column not in table.columns:
            continue
        table = table.copy()
        table["step"] = table["_logged_step"]
        if step_mapping is not None and len(step_mapping) > 0:
            mapped = table["_logged_step"].map(step_mapping)
            table.loc[mapped.notna(), "step"] = mapped
        for term_name, group in table.groupby(id_column):
            term_name = str(term_name)
            df = group[["step", "mean", "std", "count"]].dropna(subset=["step"]).copy()
            df = df.sort_values("step").reset_index(drop=True)
            df["std"] = df["std"].fillna(0)
            if term_name not in result:
                result[term_name] = {}
            result[term_name][split] = df
    return result


def plot_reward_terms_over_time(
    run_term_stats: dict[str, dict[str, pd.DataFrame]],
    step_key: str = "step",
    figsize_per_subplot: tuple[int, int] = (7, 4),
    max_cols: int = 2,
    show_ci: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot run-wise per-term reward values over time from term_summary tables.

    Args:
        run_term_stats: Dict mapping term_name -> {"train": df, "eval": df}
            (each df has columns [step, mean, std, count]).
        step_key: Label for the x-axis.
        figsize_per_subplot: Size per subplot (width, height).
        max_cols: Maximum number of columns in subplot grid.
        show_ci: Whether to show 95% CI bands.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    all_terms = sorted(run_term_stats.keys())
    if not all_terms:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, "No reward term data found", ha="center", va="center", transform=ax.transAxes)
        return fig
    n_terms = len(all_terms)
    n_cols = min(max_cols, n_terms)
    n_rows = (n_terms + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_subplot[0] * n_cols, figsize_per_subplot[1] * n_rows),
        squeeze=False,
    )
    axes_flat = axes.flatten()
    # compute global x-axis limits across all terms/splits
    all_x = [v for splits in run_term_stats.values() for df in splits.values() for v in df["step"].dropna().values]
    x_limits = _compute_percentile_x_limits(all_x, x_percentile_range) if x_percentile_range else None
    for ax_idx, term_name in enumerate(all_terms):
        ax = axes_flat[ax_idx]
        for split, df in run_term_stats[term_name].items():
            _plot_metric_series(
                ax,
                x_vals=df["step"].values,
                mean_vals=df["mean"].values,
                color=SPLIT_COLORS.get(split, TERM_COLORS[0]),
                label=f"{split} (n={len(df)})",
                std_vals=df["std"].values,
                counts=df["count"].values,
                show_ci=show_ci,
            )
        _configure_metric_ax(ax, step_key, term_name, x_limits, show_legend)
    for ax_idx in range(n_terms, len(axes_flat)):
        axes_flat[ax_idx].set_visible(False)
    return fig


# fetch run-wise term_summary tables from wandb
train_term_summary = wandb_utils.fetch_tables_with_steps(run, "train/reward/run/term_summary")
eval_term_summary = wandb_utils.fetch_tables_with_steps(run, "eval/reward/run/term_summary")
step_mapping = wandb_utils.build_step_mapping(history_df, step_key)
run_term_stats = _parse_term_summary_tables(train_term_summary, eval_term_summary, step_mapping)

if run_term_stats:
    fig = plot_reward_terms_over_time(run_term_stats, step_key=step_key)
    plt.tight_layout()
    plt.show()
else:
    print("No term_summary tables found in this run")

In [ ]:
def _fetch_reward_histograms(
    run: typing.Any,
    histogram_key: str = "reward/run/total_histogram",
    step_key: str = "train/global_step",
    history_df: pd.DataFrame | None = None,
) -> dict[str, list[dict[str, typing.Any]]]:
    """Fetch reward histogram data from wandb for train/eval splits.

    Args:
        run: The wandb Run object.
        histogram_key: Base histogram key (without train/eval prefix).
        step_key: Step key to use for x-axis (mapped from _step via history_df).
        history_df: Optional history DataFrame for _step -> step_key mapping.

    Returns:
        Dict mapping split ("train"/"eval") -> list of {step, bins, counts} dicts.
    """
    step_mapping = None
    if history_df is not None:
        step_mapping_series = wandb_utils.build_step_mapping(history_df, step_key)
        if len(step_mapping_series) > 0:
            step_mapping = step_mapping_series
    result: dict[str, list[dict[str, typing.Any]]] = {}
    for split in ("train", "eval"):
        full_key = f"{split}/{histogram_key}"
        rows = list(run.scan_history(keys=[full_key, "_step"]))
        histograms = []
        for row in rows:
            hist_data = row.get(full_key)
            wandb_step = row.get("_step")
            if hist_data is None or wandb_step is None:
                continue
            if not isinstance(hist_data, dict) or "values" not in hist_data:
                continue
            mapped_step = (
                step_mapping[wandb_step]
                if step_mapping is not None and wandb_step in step_mapping.index
                else wandb_step
            )
            if pd.isna(mapped_step):
                mapped_step = wandb_step
            histograms.append(
                {
                    "step": float(mapped_step),
                    "bins": hist_data.get("bins", []),
                    "counts": hist_data.get("values", []),
                }
            )
        if histograms:
            histograms.sort(key=lambda h: h["step"])
            result[split] = histograms
    return result


def plot_reward_histogram_heatmap(
    histograms: dict[str, list[dict[str, typing.Any]]],
    step_key: str = "train/global_step",
    figsize_per_subplot: tuple[int, int] = (8, 5),
    n_reward_bins: int = 50,
) -> matplotlib.figure.Figure | None:
    """Plot reward distribution evolution as a heatmap (step x reward -> density).

    Args:
        histograms: Dict from _fetch_reward_histograms (split -> list of histogram dicts).
        step_key: Label for the x-axis.
        figsize_per_subplot: Size per subplot.
        n_reward_bins: Number of reward bins for the y-axis.

    Returns:
        The matplotlib Figure, or None if no histogram data.
    """
    splits = [s for s in ("train", "eval") if s in histograms]
    if not splits:
        return None
    fig, axes = plt.subplots(
        1, len(splits), figsize=(figsize_per_subplot[0] * len(splits), figsize_per_subplot[1]), squeeze=False
    )
    for ax_idx, split in enumerate(splits):
        ax = axes[0, ax_idx]
        hist_list = histograms[split]
        # deduplicate histograms by step (last wins) to ensure monotonic step edges
        deduped: dict[float, dict[str, typing.Any]] = {}
        for hist in hist_list:
            deduped[hist["step"]] = hist
        hist_list = [deduped[k] for k in sorted(deduped)]
        # collect all reward values to determine global y-axis range
        all_bin_edges = []
        for hist in hist_list:
            all_bin_edges.extend(hist["bins"])
        if not all_bin_edges:
            ax.text(0.5, 0.5, "No histogram data", ha="center", va="center", transform=ax.transAxes)
            continue
        reward_min, reward_max = min(all_bin_edges), max(all_bin_edges)
        reward_edges = np.linspace(reward_min, reward_max, n_reward_bins + 1)
        # build 2D density matrix: rows=reward_bins, cols=steps
        steps = [h["step"] for h in hist_list]
        density_matrix = np.zeros((n_reward_bins, len(steps)))
        for step_idx, hist in enumerate(hist_list):
            bins = np.array(hist["bins"])
            counts = np.array(hist["counts"])
            if len(bins) == 0 or len(counts) == 0:
                continue
            # resample histogram onto uniform reward_edges grid
            bin_centers = (bins[:-1] + bins[1:]) / 2 if len(bins) > len(counts) else bins[: len(counts)]
            total = counts.sum()
            if total == 0:
                continue
            normalized = counts / total
            for orig_idx, center in enumerate(bin_centers):
                target_idx = np.searchsorted(reward_edges[1:], center, side="right")
                target_idx = min(target_idx, n_reward_bins - 1)
                density_matrix[target_idx, step_idx] += normalized[orig_idx]
        # use pcolormesh with actual step positions to avoid misleading uniform spacing
        step_arr = np.array(steps)
        step_edges = np.empty(len(step_arr) + 1)
        if len(step_arr) > 1:
            half_gaps = np.diff(step_arr) / 2
            step_edges[0] = step_arr[0] - half_gaps[0]
            step_edges[1:-1] = step_arr[:-1] + half_gaps
            step_edges[-1] = step_arr[-1] + half_gaps[-1]
        else:
            step_edges[0] = step_arr[0] - 0.5
            step_edges[1] = step_arr[0] + 0.5
        im = ax.pcolormesh(
            step_edges,
            reward_edges,
            density_matrix,
            cmap="YlOrRd",
            shading="flat",
        )
        fig.colorbar(im, ax=ax, label="Density", shrink=0.8)
        ax.set_xlabel(step_key)
        ax.set_ylabel("Reward")
        ax.set_title(f"{split.capitalize()} Reward Distribution Over Time")
    return fig


# fetch and plot reward histograms
print("Fetching reward histograms from wandb...")
reward_histograms = _fetch_reward_histograms(run, step_key=step_key, history_df=history_df)
if reward_histograms:
    for split, hists in reward_histograms.items():
        print(f"  {split}: {len(hists)} histogram snapshots")
    fig = plot_reward_histogram_heatmap(reward_histograms, step_key=step_key)
    if fig is not None:
        plt.tight_layout()
        plt.show()
else:
    print("No reward histogram data found in this run")

In [ ]:
# fetch and plot run-wise category_summary tables from wandb
train_cat_summary = wandb_utils.fetch_tables_with_steps(run, "train/reward/run/category_summary")
eval_cat_summary = wandb_utils.fetch_tables_with_steps(run, "eval/reward/run/category_summary")
run_category_stats = _parse_term_summary_tables(
    train_cat_summary,
    eval_cat_summary,
    step_mapping,
    id_column="category",
)

if run_category_stats:
    fig = plot_reward_terms_over_time(run_category_stats, step_key=step_key)
    fig.suptitle("Category-wise Reward Over Training", fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No category_summary tables found in this run")

## Parsing Statistics

In [ ]:
_STAT_SUFFIXES = ("/mean", "/std", "/count", "/min", "/max")


def _get_parsing_metrics(
    columns: list[str],
) -> dict[str, dict[str, str]]:
    """Group parsing columns by metric name and split.

    For aggregate metrics (with /mean suffix), only keeps the /mean key as primary;
    sibling std/count keys are discovered at plot time via _derive_sibling_key.

    Returns:
        Dict mapping metric_name -> {"train": key, "eval": key}.
    """
    grouped: dict[str, dict[str, str]] = {}
    for key in columns:
        if "/parsing/" not in key:
            continue
        # skip non-primary stat columns and table/summary keys
        if any(key.endswith(s) for s in _STAT_SUFFIXES[1:]):
            continue
        if "/run/" in key:
            continue
        parts = key.split("/parsing/")
        if len(parts) < 2:
            continue
        metric_name = parts[1]
        if metric_name.endswith("/mean"):
            metric_name = metric_name[:-5]
        split = "train" if "train/" in key else ("eval" if "eval/" in key else None)
        if split is None:
            continue
        if metric_name not in grouped:
            grouped[metric_name] = {}
        grouped[metric_name][split] = key
    return grouped


def plot_parsing_stats(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    figsize_per_subplot: tuple[int, int] = (7, 4),
    max_cols: int = 2,
    show_ci: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot parsing metrics over time with train/eval on the same subplot for each metric.

    For aggregate metrics (e.g. length stats with /mean suffix), CI bands are shown
    using sibling /std and /count columns when available.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        figsize_per_subplot: Size per subplot (width, height).
        max_cols: Maximum number of columns in subplot grid.
        show_ci: Whether to show CI bands (for metrics with std/count).
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    grouped = _get_parsing_metrics(list(history_df.columns))
    if not grouped:
        fig, ax = plt.subplots(figsize=(10, 4))
        ax.text(0.5, 0.5, "No parsing metrics found", ha="center", va="center", transform=ax.transAxes)
        return fig
    n_metrics = len(grouped)
    n_cols = min(max_cols, n_metrics)
    n_rows = (n_metrics + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(figsize_per_subplot[0] * n_cols, figsize_per_subplot[1] * n_rows),
        squeeze=False,
    )
    axes_flat = axes.flatten()
    # compute global x-axis limits
    all_x = []
    for keys_dict in grouped.values():
        for key in keys_dict.values():
            valid = history_df[[step_key, key]].dropna()
            if len(valid) > 0:
                all_x.extend(valid[step_key].values)
    x_limits = _compute_percentile_x_limits(all_x, x_percentile_range) if x_percentile_range else None

    # sort so ratio metrics come first, then length metrics (grouped side-by-side)
    def _metric_sort_key(name: str) -> tuple[int, str]:
        if "ratio" in name or "missing" in name:
            return (0, name)
        return (1, name)

    sorted_metrics = sorted(grouped.items(), key=lambda item: _metric_sort_key(item[0]))
    for ax_idx, (metric_name, keys_dict) in enumerate(sorted_metrics):
        ax = axes_flat[ax_idx]
        for split, key in keys_dict.items():
            if key not in history_df.columns:
                continue
            # check for sibling std/count columns
            std_key = _derive_sibling_key(key, "std")
            count_key = _derive_sibling_key(key, "count")
            has_std = std_key is not None and std_key in history_df.columns
            has_count = count_key is not None and count_key in history_df.columns
            cols = [step_key, key]
            if has_std:
                cols.append(std_key)
            if has_count:
                cols.append(count_key)
            valid_df = history_df[cols].dropna(subset=[step_key, key]).sort_values(step_key).reset_index(drop=True)
            if len(valid_df) == 0:
                continue
            _plot_metric_series(
                ax,
                x_vals=valid_df[step_key].values,
                mean_vals=valid_df[key].values,
                color=SPLIT_COLORS.get(split, TERM_COLORS[0]),
                label=f"{split} (n={len(valid_df)})",
                std_vals=valid_df[std_key].fillna(0).values if has_std else None,
                counts=valid_df[count_key].fillna(1).values if has_count else None,
                show_ci=show_ci,
            )
        ylabel = (
            "ratio"
            if ("ratio" in metric_name or "missing" in metric_name)
            else ("tokens" if "length" in metric_name else "value")
        )
        _configure_metric_ax(ax, step_key, metric_name, x_limits, show_legend)
        ax.set_ylabel(ylabel)
    for ax_idx in range(n_metrics, len(axes_flat)):
        axes_flat[ax_idx].set_visible(False)
    return fig


fig = plot_parsing_stats(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

## Sample Output Viewer

In [ ]:
def normalize_rewards_table(
    df: pd.DataFrame,
    use_logged_step_as_step: bool = False,
) -> pd.DataFrame:
    """Normalize rewards table to consistent schema.

    Expected columns from WandBRewardLogger (see pyine/organisms/models/rewards/core/logging.py):
        sample_id, generation_count, batch_count, local_batch_idx, completion_idx, rank, step,
        prompt, expected_output, model_output, reasoning, final_answer, reward_total,
        reward_terms_json, reward_terms_raw_json, reward_metrics_json, categories_json, tags_json,
        difficulty_source, difficulty_score, difficulty_bin, difficulty_raw_primary,
        difficulty_secondary_json, predict_type, code_type, has_code_override

    Args:
        df: Raw rewards table DataFrame.
        use_logged_step_as_step: If True, use _logged_step (the W&B step when the table was
            flushed) as the authoritative step value instead of the logged 'step' column.
            Defaults to False (use the logged 'step' value).

    Returns:
        Normalized DataFrame with parsed JSON columns and consistent schema.
    """
    df = df.copy()
    # parse JSON columns
    json_columns = [
        "reward_terms_json",
        "reward_terms_raw_json",
        "reward_metrics_json",
        "categories_json",
        "tags_json",
        "difficulty_secondary_json",
    ]
    for col in json_columns:
        if col in df.columns and df[col].dtype == object:
            df[col] = df[col].apply(lambda x: json.loads(x) if isinstance(x, str) else x)
    # ensure all expected columns exist (matches WandBRewardLogger schema)
    expected_cols = [
        "sample_id",
        "generation_count",
        "batch_count",
        "local_batch_idx",
        "completion_idx",
        "rank",
        "step",
        "prompt",
        "expected_output",
        "model_output",
        "reasoning",
        "final_answer",
        "reward_total",
        "reward_terms_json",
        "reward_terms_raw_json",
        "reward_metrics_json",
        "categories_json",
        "tags_json",
        "difficulty_source",
        "difficulty_score",
        "difficulty_bin",
        "difficulty_raw_primary",
        "difficulty_secondary_json",
        "predict_type",
        "code_type",
        "has_code_override",
    ]
    for col in expected_cols:
        if col not in df.columns:
            df[col] = None
    # optionally use _logged_step as the authoritative step (added by fetch_tables_with_steps)
    if use_logged_step_as_step and "_logged_step" in df.columns:
        df["step"] = df["_logged_step"].fillna(df["step"])
    # sort by step numerically (ensure numeric type for proper sorting)
    if "step" in df.columns:
        df["step"] = pd.to_numeric(df["step"], errors="coerce")
        df = df.sort_values("step", na_position="last").reset_index(drop=True)
    return df


# fetch tables with their true logged steps from W&B
rewards_table = wandb_utils.fetch_tables_with_steps(run, "train/generation_details")
if rewards_table is not None:
    rewards_table = normalize_rewards_table(rewards_table)
    print(f"Loaded {len(rewards_table)} sample records from rewards table")
    print(f"Columns: {list(rewards_table.columns)}")
    if "_logged_step" in rewards_table.columns:
        unique_steps = rewards_table["_logged_step"].nunique()
        print(f"Samples from {unique_steps} unique logged steps")
else:
    print("No rewards table found (log_tables may be disabled in reward logging config)")

In [ ]:
def get_parsing_metric(row: pd.Series, key: str, default: int = 0) -> int:
    """Get a parsing metric from reward_metrics_json (handles prefixed keys)."""
    metrics = row.get("reward_metrics_json")
    if isinstance(metrics, dict):
        # try exact key first, then search for keys ending with the expected suffix
        if key in metrics:
            return metrics[key]
        suffix = f"parsing/{key}"
        for k, v in metrics.items():
            if k.endswith(suffix):
                return v
    return default


def display_sample(row: pd.Series) -> None:
    """Display a single sample from the rewards table."""
    # check for missing fields using actual values
    reasoning = row.get("reasoning")
    has_reasoning = bool(reasoning) if isinstance(reasoning, str) else False
    has_answer = get_parsing_metric(row, "has_answer", default=1) == 1
    # build status indicators
    status_parts = []
    if not has_reasoning:
        status_parts.append('<span style="color: orange; font-weight: bold;">⚠ NO REASONING</span>')
    if not has_answer:
        status_parts.append('<span style="color: orange; font-weight: bold;">⚠ NO ANSWER</span>')
    status_html = " ".join(status_parts) if status_parts else '<span style="color: green;">✓ OK</span>'
    html_parts = []
    # header with sample_id and status
    sample_id = row.get("sample_id", "N/A")
    html_parts.append(f"<h4>Sample: {sample_id} {status_html}</h4>")
    # metadata table
    html_parts.append("<table style='border-collapse: collapse; margin-bottom: 10px;'>")
    # step info
    step = row.get("step")
    logged_step = row.get("_logged_step")
    step_str = f"{int(step)}" if pd.notna(step) else "N/A"
    if pd.notna(logged_step) and logged_step != step:
        step_str += f" (logged at W&B step {int(logged_step)})"
    html_parts.append(f"<tr><td><b>Step:</b></td><td>{step_str}</td></tr>")
    # generation info
    batch_idx = row.get("local_batch_idx")
    completion_idx = row.get("completion_idx")
    rank = row.get("rank")
    gen_count = row.get("generation_count")
    batch_count = row.get("batch_count")
    if (
        pd.notna(batch_idx)
        or pd.notna(gen_count)
        or pd.notna(batch_count)
        or pd.notna(completion_idx)
        or pd.notna(rank)
    ):
        gen_str_parts = []
        if pd.notna(rank):
            gen_str_parts.append(f"rank={int(rank)}")
        if pd.notna(batch_idx):
            gen_str_parts.append(f"batch_idx={int(batch_idx)}")
        if pd.notna(completion_idx):
            gen_str_parts.append(f"completion={int(completion_idx)}")
        if pd.notna(batch_count):
            gen_str_parts.append(f"batch={int(batch_count)}")
        if pd.notna(gen_count):
            gen_str_parts.append(f"gen_count={int(gen_count)}")
        html_parts.append(f"<tr><td><b>Generation:</b></td><td>{', '.join(gen_str_parts)}</td></tr>")
    # reward
    reward_total = row.get("reward_total")
    reward_str = f"{reward_total:.4f}" if pd.notna(reward_total) else "N/A"
    html_parts.append(f"<tr><td><b>Reward:</b></td><td>{reward_str}</td></tr>")
    # categories
    cats = row.get("categories_json", [])
    if cats and isinstance(cats, list):
        html_parts.append(f"<tr><td><b>Categories:</b></td><td>{', '.join(cats)}</td></tr>")
    # tags
    tags = row.get("tags_json", [])
    if tags and isinstance(tags, list):
        html_parts.append(f"<tr><td><b>Tags:</b></td><td>{', '.join(tags)}</td></tr>")
    # predict_type and code_type
    predict_type = row.get("predict_type")
    code_type = row.get("code_type")
    has_override = row.get("has_code_override")
    if predict_type or code_type:
        type_parts = []
        if predict_type:
            type_parts.append(f"predict={predict_type}")
        if code_type:
            code_str = str(code_type)
            if has_override:
                code_str += " (override)"
            type_parts.append(f"code={code_str}")
        html_parts.append(f"<tr><td><b>Type:</b></td><td>{', '.join(type_parts)}</td></tr>")
    # difficulty
    diff_source = row.get("difficulty_source")
    diff_score = row.get("difficulty_score")
    diff_bin = row.get("difficulty_bin")
    if diff_source or pd.notna(diff_score):
        diff_parts = []
        if diff_source:
            diff_parts.append(str(diff_source))
        if pd.notna(diff_score):
            diff_parts.append(f"score={diff_score:.3f}")
        if pd.notna(diff_bin):
            diff_parts.append(f"bin={int(diff_bin)}")
        diff_raw = row.get("difficulty_raw_primary")
        if pd.notna(diff_raw):
            diff_parts.append(f"raw={diff_raw:.3f}")
        html_parts.append(f"<tr><td><b>Difficulty:</b></td><td>{', '.join(diff_parts)}</td></tr>")
    html_parts.append("</table>")
    # reward terms
    terms = row.get("reward_terms_json", {})
    if terms and isinstance(terms, dict):
        terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in terms.items())
        html_parts.append(f"<b>Reward Terms:</b> {terms_str}<br>")
    # raw reward terms (if available and different from terms)
    raw_terms = row.get("reward_terms_raw_json", {})
    if raw_terms and isinstance(raw_terms, dict) and raw_terms != terms:
        raw_terms_str = ", ".join(f"{k}: {v:.3f}" for k, v in raw_terms.items())
        html_parts.append(f"<b>Raw Terms:</b> {raw_terms_str}<br>")
    html_parts.append("<hr>")
    # prompt (escape HTML to show raw tags like <final>)
    pre_style = "white-space: pre-wrap; max-height: 300px; overflow-y: auto;"
    prompt = row.get("prompt") or ""
    html_parts.append(f"<b>Prompt:</b><br><pre style='{pre_style}'>{html.escape(prompt)}</pre>")
    # expected output
    expected = row.get("expected_output")
    if expected:
        html_parts.append(
            f"<b>Expected Output:</b><br><pre style='white-space: pre-wrap;'>{html.escape(expected)}</pre>"
        )
    # model output (escape HTML to show raw tags like <final>)
    pre_style_tall = "white-space: pre-wrap; max-height: 400px; overflow-y: auto;"
    output = row.get("model_output") or ""
    html_parts.append(f"<b>Model Output:</b><br><pre style='{pre_style_tall}'>{html.escape(output)}</pre>")
    # reasoning (only show separately if different from model_output)
    if has_reasoning and reasoning != output:
        html_parts.append(f"<b>Reasoning:</b><br><pre style='{pre_style}'>{html.escape(reasoning)}</pre>")
    elif not has_reasoning:
        html_parts.append('<b>Reasoning:</b> <span style="color: orange;">(missing)</span><br>')
    # final answer
    answer = row.get("final_answer")
    if answer:
        html_parts.append(f"<b>Final Answer:</b><br><pre style='white-space: pre-wrap;'>{html.escape(answer)}</pre>")
    elif not has_answer:
        html_parts.append('<b>Final Answer:</b> <span style="color: orange;">(missing)</span><br>')
    ipy_display.display(ipy_display.HTML("".join(html_parts)))


if rewards_table is not None and len(rewards_table) > 0:
    print(f"Next cell can display up to {len(rewards_table)} samples.")
else:
    print("No samples to display")

In [ ]:
# interactive sample browser using ipywidgets
try:
    import ipywidgets

    WIDGETS_AVAILABLE = True
except ImportError:
    WIDGETS_AVAILABLE = False
    print("ipywidgets not available - interactive browser disabled")


def create_sample_browser(
    rewards_table: pd.DataFrame,
    run_url: str,
) -> "ipywidgets.VBox":
    """Create an interactive sample browser widget."""
    current_df = rewards_table.copy()
    # filter controls — checkboxes
    show_missing_answer = ipywidgets.Checkbox(value=False, description="Missing answer")
    show_missing_reasoning = ipywidgets.Checkbox(value=False, description="Missing reasoning")
    # filter controls — dropdowns for predict_type, code_type
    all_label = "(all)"
    predict_types = (
        sorted(rewards_table["predict_type"].dropna().unique()) if "predict_type" in rewards_table.columns else []
    )
    code_types = sorted(rewards_table["code_type"].dropna().unique()) if "code_type" in rewards_table.columns else []
    predict_dropdown = ipywidgets.Dropdown(
        options=[all_label] + list(predict_types),
        value=all_label,
        description="Predict:",
    )
    code_dropdown = ipywidgets.Dropdown(
        options=[all_label] + list(code_types),
        value=all_label,
        description="Code type:",
    )
    # navigation
    idx_slider = ipywidgets.IntSlider(value=0, min=0, max=max(0, len(current_df) - 1), description="Index:")
    prev_btn = ipywidgets.Button(description="< Prev")
    next_btn = ipywidgets.Button(description="Next >")
    # output area
    output = ipywidgets.Output()
    info_label = ipywidgets.HTML(value=f"Total samples: {len(current_df)}")

    def apply_filters() -> None:
        nonlocal current_df
        df = rewards_table.copy()
        if show_missing_answer.value:
            df = df[df.apply(lambda r: get_parsing_metric(r, "has_answer", default=1) == 0, axis=1)]
        if show_missing_reasoning.value:
            df = df[df["reasoning"].apply(lambda x: not x if isinstance(x, str) else True)]
        if predict_dropdown.value != all_label and "predict_type" in df.columns:
            df = df[df["predict_type"] == predict_dropdown.value]
        if code_dropdown.value != all_label and "code_type" in df.columns:
            df = df[df["code_type"] == code_dropdown.value]
        current_df = df
        idx_slider.max = max(0, len(current_df) - 1)
        idx_slider.value = min(idx_slider.value, idx_slider.max)
        info_label.value = f"Showing {len(current_df)} of {len(rewards_table)} samples"
        update_display(None)

    def update_display(_: typing.Any) -> None:
        output.clear_output()
        with output:
            if len(current_df) == 0:
                print("No samples match current filters")
                return
            idx = idx_slider.value
            if idx < len(current_df):
                display_sample(current_df.iloc[idx])

    def on_prev(_: typing.Any) -> None:
        if idx_slider.value > 0:
            idx_slider.value -= 1

    def on_next(_: typing.Any) -> None:
        if idx_slider.value < idx_slider.max:
            idx_slider.value += 1

    # connect handlers
    idx_slider.observe(update_display, names="value")
    prev_btn.on_click(on_prev)
    next_btn.on_click(on_next)
    show_missing_answer.observe(lambda _: apply_filters(), names="value")
    show_missing_reasoning.observe(lambda _: apply_filters(), names="value")
    predict_dropdown.observe(lambda _: apply_filters(), names="value")
    code_dropdown.observe(lambda _: apply_filters(), names="value")
    # initial display
    update_display(None)
    # layout
    checkbox_row = ipywidgets.HBox([show_missing_answer, show_missing_reasoning])
    dropdown_row = ipywidgets.HBox([predict_dropdown, code_dropdown])
    nav_box = ipywidgets.HBox([prev_btn, idx_slider, next_btn])
    url_html = ipywidgets.HTML(value=f"<a href='{run_url}' target='_blank'>Open in W&B</a>")
    return ipywidgets.VBox([info_label, checkbox_row, dropdown_row, nav_box, url_html, output])


if WIDGETS_AVAILABLE and rewards_table is not None and len(rewards_table) > 0:
    browser = create_sample_browser(rewards_table, run.url)
    ipy_display.display(browser)
elif rewards_table is None:
    print("No rewards table available for browsing")
else:
    print("No samples in rewards table")

In [ ]:
# sample filtering configuration
FILTER_STEP_RANGE = (None, None)  # (min_step, max_step) or None
FILTER_REWARD_RANGE = (None, None)  # (min_reward, max_reward) or None
FILTER_CATEGORIES: list[str] = []  # list of category strings to include (empty = all)
FILTER_PREDICT_TYPES: list[str] = []  # list of predict_type values to include (empty = all)
FILTER_CODE_TYPES: list[str] = []  # list of code_type values to include (empty = all)
FILTER_DIFFICULTY_RANGE = (None, None)  # (min_score, max_score) or None


def filter_samples(
    rewards_table: pd.DataFrame,
    step_range: tuple[int | None, int | None] = (None, None),
    reward_range: tuple[float | None, float | None] = (None, None),
    categories: list[str] | None = None,
    predict_types: list[str] | None = None,
    code_types: list[str] | None = None,
    difficulty_range: tuple[float | None, float | None] = (None, None),
) -> pd.DataFrame:
    """Filter samples by step, reward, category, type, or difficulty."""
    df = rewards_table.copy()
    # step range filter
    if step_range[0] is not None and "step" in df.columns:
        df = df[df["step"] >= step_range[0]]
    if step_range[1] is not None and "step" in df.columns:
        df = df[df["step"] <= step_range[1]]
    # reward range filter
    if reward_range[0] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] >= reward_range[0]]
    if reward_range[1] is not None and "reward_total" in df.columns:
        df = df[df["reward_total"] <= reward_range[1]]
    # category filter
    if categories and "categories_json" in df.columns:

        def has_category(cats: typing.Any) -> bool:
            if not isinstance(cats, list):
                return False
            return any(c in cats for c in categories)

        df = df[df["categories_json"].apply(has_category)]
    # predict_type filter
    if predict_types and "predict_type" in df.columns:
        df = df[df["predict_type"].isin(predict_types)]
    # code_type filter
    if code_types and "code_type" in df.columns:
        df = df[df["code_type"].isin(code_types)]
    # difficulty score range filter
    if difficulty_range[0] is not None and "difficulty_score" in df.columns:
        df = df[df["difficulty_score"] >= difficulty_range[0]]
    if difficulty_range[1] is not None and "difficulty_score" in df.columns:
        df = df[df["difficulty_score"] <= difficulty_range[1]]
    return df


if rewards_table is not None:
    filtered = filter_samples(
        rewards_table,
        FILTER_STEP_RANGE,
        FILTER_REWARD_RANGE,
        FILTER_CATEGORIES,
        FILTER_PREDICT_TYPES,
        FILTER_CODE_TYPES,
        FILTER_DIFFICULTY_RANGE,
    )
    print(f"Filtered to {len(filtered)} samples (from {len(rewards_table)} total)")
else:
    filtered = None
    print("No rewards table to filter")

In [ ]:
def plot_reward_distributions(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 10),
) -> matplotlib.figure.Figure:
    """Plot reward distributions."""
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    # 1. total reward histogram
    ax = axes[0, 0]
    if "reward_total" in rewards_table.columns:
        rewards = rewards_table["reward_total"].dropna()
        ax.hist(rewards, bins=50, color="#2C7BB6", alpha=0.7, edgecolor="black")
        ax.axvline(rewards.mean(), color="red", linestyle="--", label=f"Mean: {rewards.mean():.3f}")
        ax.set_xlabel("Reward")
        ax.set_ylabel("Count")
        ax.legend()
    ax.set_title("Total Reward Distribution")
    ax.grid(True, alpha=0.3)
    # 2. reward over steps (percentile bands)
    ax = axes[0, 1]
    if "step" in rewards_table.columns and "reward_total" in rewards_table.columns:
        df_valid = rewards_table[["step", "reward_total"]].dropna().sort_values("step")
        if len(df_valid) > 0:
            # bin samples into ~30 equal-count windows for smooth percentile curves
            n_bins = min(30, len(df_valid) // 5) if len(df_valid) >= 10 else 1
            df_valid = df_valid.copy()
            df_valid["_bin"] = pd.qcut(df_valid["step"], q=n_bins, duplicates="drop")
            grouped = df_valid.groupby("_bin", observed=True)["reward_total"]
            bin_centers = df_valid.groupby("_bin", observed=True)["step"].mean()
            median = grouped.median()
            p25, p75 = grouped.quantile(0.25), grouped.quantile(0.75)
            p10, p90 = grouped.quantile(0.10), grouped.quantile(0.90)
            # faded scatter for raw data
            ax.scatter(df_valid["step"], df_valid["reward_total"], alpha=0.08, s=6, c="#888888", zorder=1)
            # percentile bands
            ax.fill_between(bin_centers, p10, p90, alpha=0.15, color="#2C7BB6", label="10th-90th pct", zorder=2)
            ax.fill_between(bin_centers, p25, p75, alpha=0.3, color="#2C7BB6", label="25th-75th pct", zorder=3)
            ax.plot(bin_centers, median, color="#2C7BB6", linewidth=2, label="Median", zorder=4)
            ax.legend(fontsize=8)
    ax.set_title("Reward vs Step")
    ax.set_xlabel("Step")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    # 3. reward terms breakdown (if available)
    ax = axes[1, 0]
    if "reward_terms_json" in rewards_table.columns:
        term_data: dict[str, list[float]] = {}
        for terms in rewards_table["reward_terms_json"].dropna():
            if isinstance(terms, dict):
                for term, value in terms.items():
                    if term not in term_data:
                        term_data[term] = []
                    term_data[term].append(value)
        if term_data:
            term_names = sorted(term_data.keys())
            term_values = [term_data[t] for t in term_names]
            parts = ax.violinplot(term_values, positions=range(len(term_names)), showmedians=True, showextrema=False)
            for pc in parts["bodies"]:
                pc.set_facecolor("#2C7BB6")
                pc.set_alpha(0.4)
            parts["cmedians"].set_color("#D7191C")
            # jittered strip overlay (subsample if too many points)
            for term_idx, vals in enumerate(term_values):
                sample = (
                    vals
                    if len(vals) <= 200
                    else [vals[j] for j in np.random.default_rng(0).choice(len(vals), 200, replace=False)]
                )
                jitter = np.random.default_rng(term_idx).uniform(-0.15, 0.15, len(sample))
                ax.scatter(term_idx + jitter, sample, alpha=0.15, s=6, c="#333333", zorder=2)
            ax.set_xticks(range(len(term_names)))
            ax.set_xticklabels(term_names, rotation=45, ha="right")
    ax.set_title("Reward Terms Distribution")
    ax.set_ylabel("Value")
    ax.grid(True, alpha=0.3)
    # 4. rewards by category (if available)
    ax = axes[1, 1]
    if "categories_json" in rewards_table.columns and "reward_total" in rewards_table.columns:
        cat_rewards: dict[str, list[float]] = {}
        for _idx, row in rewards_table.iterrows():
            cats = row.get("categories_json", [])
            reward = row.get("reward_total")
            if isinstance(cats, list) and cats and reward is not None:
                cat = cats[0]  # use first category
                if cat not in cat_rewards:
                    cat_rewards[cat] = []
                cat_rewards[cat].append(reward)
        if cat_rewards:
            cat_names = sorted(cat_rewards.keys())[:8]
            cat_values = [cat_rewards[c] for c in cat_names]
            parts = ax.violinplot(cat_values, positions=range(len(cat_names)), showmedians=True, showextrema=False)
            for pc in parts["bodies"]:
                pc.set_facecolor("#1A9641")
                pc.set_alpha(0.4)
            parts["cmedians"].set_color("#D7191C")
            for cat_idx, vals in enumerate(cat_values):
                sample = (
                    vals
                    if len(vals) <= 200
                    else [vals[j] for j in np.random.default_rng(0).choice(len(vals), 200, replace=False)]
                )
                jitter = np.random.default_rng(cat_idx).uniform(-0.15, 0.15, len(sample))
                ax.scatter(cat_idx + jitter, sample, alpha=0.15, s=6, c="#333333", zorder=2)
            ax.set_xticks(range(len(cat_names)))
            ax.set_xticklabels(cat_names, rotation=45, ha="right", fontsize=8)
    ax.set_title("Rewards by Category (first category per sample)")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_reward_distributions(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for distribution analysis")

In [ ]:
def plot_length_vs_reward(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 5),
) -> matplotlib.figure.Figure:
    """Scatter plot: output/reasoning length vs reward."""
    fig, axes = plt.subplots(1, 2, figsize=figsize)
    # compute lengths if not already present
    df = rewards_table.copy()
    if "output_length" not in df.columns and "model_output" in df.columns:
        df["output_length"] = df["model_output"].apply(lambda x: len(str(x)) if x else 0)
    if "reasoning_length" not in df.columns and "reasoning" in df.columns:
        df["reasoning_length"] = df["reasoning"].apply(lambda x: len(str(x)) if x else 0)
    # output length vs reward
    ax = axes[0]
    if "output_length" in df.columns and "reward_total" in df.columns:
        valid = df[["output_length", "reward_total"]].dropna()
        ax.scatter(valid["output_length"], valid["reward_total"], alpha=0.2, s=8, c="#2C7BB6")
        if len(valid) > 10:
            z = np.polyfit(valid["output_length"], valid["reward_total"], 1)
            p = np.poly1d(z)
            x_line = np.linspace(valid["output_length"].min(), valid["output_length"].max(), 100)
            ax.plot(x_line, p(x_line), "r--", alpha=0.6, label=f"All (n={len(valid)})")
        positive = valid[valid["reward_total"] != 0]
        if len(positive) > 10:
            z = np.polyfit(positive["output_length"], positive["reward_total"], 1)
            p = np.poly1d(z)
            x_line = np.linspace(positive["output_length"].min(), positive["output_length"].max(), 100)
            ax.plot(x_line, p(x_line), "g-", alpha=0.7, linewidth=1.5, label=f"Non-zero (n={len(positive)})")
        ax.legend(fontsize=8)
    ax.set_xlabel("Output Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Output Length vs Reward")
    ax.grid(True, alpha=0.3)
    # reasoning length vs reward
    ax = axes[1]
    if "reasoning_length" in df.columns and "reward_total" in df.columns:
        valid = df[["reasoning_length", "reward_total"]].dropna()
        valid = valid[valid["reasoning_length"] > 0]  # only samples with reasoning
        if len(valid) > 0:
            ax.scatter(valid["reasoning_length"], valid["reward_total"], alpha=0.2, s=8, c="#7570B3")
            if len(valid) > 10:
                z = np.polyfit(valid["reasoning_length"], valid["reward_total"], 1)
                p = np.poly1d(z)
                x_line = np.linspace(valid["reasoning_length"].min(), valid["reasoning_length"].max(), 100)
                ax.plot(x_line, p(x_line), "r--", alpha=0.6, label=f"All (n={len(valid)})")
            positive = valid[valid["reward_total"] != 0]
            if len(positive) > 10:
                z = np.polyfit(positive["reasoning_length"], positive["reward_total"], 1)
                p = np.poly1d(z)
                x_line = np.linspace(positive["reasoning_length"].min(), positive["reasoning_length"].max(), 100)
                ax.plot(x_line, p(x_line), "g-", alpha=0.7, linewidth=1.5, label=f"Non-zero (n={len(positive)})")
            ax.legend(fontsize=8)
        else:
            ax.text(
                0.5,
                0.5,
                "No samples with reasoning",
                ha="center",
                va="center",
                transform=ax.transAxes,
            )
    ax.set_xlabel("Reasoning Length (chars)")
    ax.set_ylabel("Reward")
    ax.set_title("Reasoning Length vs Reward")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_length_vs_reward(rewards_table)
    plt.tight_layout()
    plt.show()
else:
    print("No rewards table available for length analysis")

## Reward vs Difficulty Analysis

In [ ]:
def plot_reward_vs_difficulty(
    rewards_table: pd.DataFrame,
    figsize: tuple[int, int] = (14, 10),
    n_percentile_bins: int = 30,
) -> matplotlib.figure.Figure | None:
    """Plot reward vs difficulty analysis (score, bins, trends over time).

    Args:
        rewards_table: Normalized rewards DataFrame with difficulty columns.
        figsize: Figure size.
        n_percentile_bins: Number of bins for percentile band plots.

    Returns:
        The matplotlib Figure, or None if no difficulty data is available.
    """
    has_score = "difficulty_score" in rewards_table.columns and rewards_table["difficulty_score"].notna().any()
    has_bin = "difficulty_bin" in rewards_table.columns and rewards_table["difficulty_bin"].notna().any()
    if not has_score and not has_bin:
        return None
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    # 1. reward vs difficulty_score (percentile bands)
    ax = axes[0, 0]
    if has_score and "reward_total" in rewards_table.columns:
        df_valid = rewards_table[["difficulty_score", "reward_total"]].dropna().sort_values("difficulty_score")
        if len(df_valid) >= 10:
            n_bins = min(n_percentile_bins, len(df_valid) // 5)
            df_valid = df_valid.copy()
            df_valid["_bin"] = pd.qcut(df_valid["difficulty_score"], q=n_bins, duplicates="drop")
            grouped = df_valid.groupby("_bin", observed=True)["reward_total"]
            bin_centers = df_valid.groupby("_bin", observed=True)["difficulty_score"].mean()
            median = grouped.median()
            p25, p75 = grouped.quantile(0.25), grouped.quantile(0.75)
            p10, p90 = grouped.quantile(0.10), grouped.quantile(0.90)
            ax.scatter(df_valid["difficulty_score"], df_valid["reward_total"], alpha=0.08, s=6, c="#888888", zorder=1)
            ax.fill_between(bin_centers, p10, p90, alpha=0.15, color="#2C7BB6", label="10th-90th pct", zorder=2)
            ax.fill_between(bin_centers, p25, p75, alpha=0.3, color="#2C7BB6", label="25th-75th pct", zorder=3)
            ax.plot(bin_centers, median, color="#2C7BB6", linewidth=2, label="Median", zorder=4)
            ax.legend(fontsize=8)
        elif len(df_valid) > 0:
            ax.scatter(df_valid["difficulty_score"], df_valid["reward_total"], alpha=0.4, s=10, c="#2C7BB6")
    ax.set_title("Reward vs Difficulty Score")
    ax.set_xlabel("Difficulty Score")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    # 2. reward distribution by difficulty_bin (violin + strip)
    ax = axes[0, 1]
    if has_bin and "reward_total" in rewards_table.columns:
        df_valid = rewards_table[["difficulty_bin", "reward_total"]].dropna()
        if len(df_valid) > 0:
            bins = sorted(df_valid["difficulty_bin"].unique())
            bin_labels = [str(int(b)) for b in bins]
            bin_values = [df_valid.loc[df_valid["difficulty_bin"] == b, "reward_total"].values for b in bins]
            # only plot bins with data
            non_empty = [(lbl, vals) for lbl, vals in zip(bin_labels, bin_values, strict=False) if len(vals) > 0]
            if non_empty:
                ne_labels, ne_values = zip(*non_empty, strict=False)
                parts = ax.violinplot(ne_values, positions=range(len(ne_labels)), showmedians=True, showextrema=False)
                for pc in parts["bodies"]:
                    pc.set_facecolor("#9970AB")
                    pc.set_alpha(0.4)
                parts["cmedians"].set_color("#D7191C")
                for bin_idx, vals in enumerate(ne_values):
                    sample = (
                        vals
                        if len(vals) <= 200
                        else vals[np.random.default_rng(0).choice(len(vals), 200, replace=False)]
                    )
                    jitter = np.random.default_rng(bin_idx).uniform(-0.15, 0.15, len(sample))
                    ax.scatter(bin_idx + jitter, sample, alpha=0.15, s=6, c="#333333", zorder=2)
                ax.set_xticks(range(len(ne_labels)))
                ax.set_xticklabels(ne_labels)
    ax.set_title("Reward by Difficulty Bin")
    ax.set_xlabel("Difficulty Bin")
    ax.set_ylabel("Reward")
    ax.grid(True, alpha=0.3)
    # 3. reward trend over steps by difficulty_bin (heatmap)
    ax = axes[1, 0]
    if has_bin and "step" in rewards_table.columns and "reward_total" in rewards_table.columns:
        df_valid = rewards_table[["step", "difficulty_bin", "reward_total"]].dropna()
        if len(df_valid) >= 10:
            diff_bins = sorted(df_valid["difficulty_bin"].unique())
            n_step_bins = min(20, len(df_valid["step"].unique()))
            df_valid = df_valid.copy()
            df_valid["_step_bin"] = pd.qcut(df_valid["step"], q=n_step_bins, duplicates="drop")
            step_labels = df_valid.groupby("_step_bin", observed=True)["step"].mean()
            pivot = df_valid.pivot_table(
                values="reward_total",
                index="difficulty_bin",
                columns="_step_bin",
                aggfunc="median",
            )
            pivot = pivot.reindex(diff_bins)
            im = ax.imshow(
                pivot.values,
                aspect="auto",
                cmap="RdYlGn",
                origin="lower",
                vmin=pivot.values[np.isfinite(pivot.values)].min() if np.any(np.isfinite(pivot.values)) else 0,
                vmax=pivot.values[np.isfinite(pivot.values)].max() if np.any(np.isfinite(pivot.values)) else 1,
            )
            # x ticks: show ~5 evenly spaced step labels
            n_xticks = min(5, len(step_labels))
            xtick_positions = np.linspace(0, len(step_labels) - 1, n_xticks, dtype=int)
            ax.set_xticks(xtick_positions)
            ax.set_xticklabels([f"{step_labels.iloc[p]:.0f}" for p in xtick_positions], fontsize=8)
            # y ticks: difficulty bins
            ax.set_yticks(range(len(diff_bins)))
            ax.set_yticklabels([f"{int(b)}" for b in diff_bins], fontsize=8)
            fig.colorbar(im, ax=ax, label="Median Reward", shrink=0.8)
    ax.set_title("Median Reward: Step x Difficulty Bin")
    ax.set_xlabel("Step")
    ax.set_ylabel("Difficulty Bin")
    # 4. sample count per difficulty_bin
    ax = axes[1, 1]
    if has_bin:
        df_valid = rewards_table["difficulty_bin"].dropna()
        if len(df_valid) > 0:
            bin_counts = df_valid.value_counts().sort_index()
            bin_labels = [str(int(b)) for b in bin_counts.index]
            bars = ax.bar(range(len(bin_labels)), bin_counts.values, color="#FDAE61", edgecolor="#333333", alpha=0.8)
            ax.set_xticks(range(len(bin_labels)))
            ax.set_xticklabels(bin_labels)
            for bar, count in zip(bars, bin_counts.values, strict=False):
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height(),
                    str(count),
                    ha="center",
                    va="bottom",
                    fontsize=7,
                )
    ax.set_title("Samples per Difficulty Bin")
    ax.set_xlabel("Difficulty Bin")
    ax.set_ylabel("Count")
    ax.grid(True, alpha=0.3)
    return fig


if rewards_table is not None and len(rewards_table) > 0:
    fig = plot_reward_vs_difficulty(rewards_table)
    if fig is not None:
        plt.tight_layout()
        plt.show()
    else:
        print("No difficulty data available in rewards table")
else:
    print("No rewards table available for difficulty analysis")

## TRL Metrics

In [ ]:
def plot_trl_metrics(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    figsize: tuple[int, int] = (14, 10),
    show_scatter: bool = True,
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot TRL-related metrics: entropy, KL divergence, clip ratios, and loss.

    Shows train/eval curves together where available. Metrics without data show a placeholder.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        figsize: Figure size.
        show_scatter: Whether to show individual data points.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure object.
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    # define metrics to plot: (title, ylabel, key_patterns)
    # pick one clip ratio key per split: prefer region_mean, fall back to bare clip_ratio
    clip_ratio_keys = []
    for prefix in ("train", "eval"):
        region_key = f"{prefix}/clip_ratio/region_mean"
        bare_key = f"{prefix}/clip_ratio"
        if region_key in history_df.columns:
            clip_ratio_keys.append(region_key)
        elif bare_key in history_df.columns:
            clip_ratio_keys.append(bare_key)
    if not clip_ratio_keys and "clip_ratio" in history_df.columns:
        clip_ratio_keys.append("clip_ratio")
    loss_keys = sorted(k for k in history_df.columns if k.endswith("/loss") or k == "loss")
    metrics_config = [
        ("Entropy", "entropy", ["train/entropy", "eval/entropy"]),
        ("KL Divergence", "kl", ["train/kl", "eval/kl", "kl"]),
        ("Clip Ratio", "clip_ratio", clip_ratio_keys),
        ("Loss", "loss", loss_keys),
    ]
    # compute global x-axis limits
    all_x_values: list[float] = []
    for _, _, keys in metrics_config:
        for key in keys:
            if key in history_df.columns:
                valid_df = history_df[[step_key, key]].dropna()
                if len(valid_df) > 0:
                    all_x_values.extend(valid_df[step_key].values)
    x_limits = (
        _compute_percentile_x_limits(all_x_values, x_percentile_range) if x_percentile_range and all_x_values else None
    )
    # plot each metric
    axes_flat = axes.flatten()
    for ax_idx, (title, ylabel, candidate_keys) in enumerate(metrics_config):
        ax = axes_flat[ax_idx]
        # filter to keys that exist in history_df
        valid_keys = [k for k in candidate_keys if k in history_df.columns]
        if not valid_keys:
            ax.text(0.5, 0.5, f"No {ylabel} metrics found", ha="center", va="center", transform=ax.transAxes)
            ax.set_title(title)
            ax.set_xlabel(step_key)
            ax.set_ylabel(ylabel)
            ax.grid(axis="y", alpha=0.3)
            continue
        plotted_any = False
        for key in valid_keys:
            valid_df = history_df[[step_key, key]].dropna().copy()
            if len(valid_df) == 0:
                continue
            valid_df = valid_df.sort_values(step_key).reset_index(drop=True)
            x_vals = valid_df[step_key].values
            y_vals = valid_df[key].values
            # determine color based on split
            if "train/" in key:
                color = SPLIT_COLORS["train"]
                label = f"train (n={len(valid_df)})"
            elif "eval/" in key:
                color = SPLIT_COLORS["eval"]
                label = f"eval (n={len(valid_df)})"
            else:
                color = TERM_COLORS[2]
                short_key = key.split("/")[-1] if "/" in key else key
                label = f"{short_key} (n={len(valid_df)})"
            ax.plot(x_vals, y_vals, color=color, linewidth=1.5, alpha=0.8)
            if show_scatter:
                ax.scatter(x_vals, y_vals, color=color, s=12, alpha=0.4, label=label)
            else:
                ax.plot([], [], color=color, label=label)  # for legend
            plotted_any = True
        if not plotted_any:
            ax.text(0.5, 0.5, f"{ylabel} keys found but no data", ha="center", va="center", transform=ax.transAxes)
        ax.set_xlabel(step_key)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        if x_limits is not None:
            ax.set_xlim(*x_limits)
        ax.grid(axis="y", alpha=0.3)
        if show_legend and plotted_any:
            ax.legend(loc="best", fontsize=8)
    return fig


fig = plot_trl_metrics(history_df, step_key=step_key)
plt.tight_layout()
plt.show()

# print data availability summary
print("\nTRL Metrics Data Availability:")
trl_keys = [k for k in history_df.columns if any(m in k for m in ["entropy", "kl", "clip_ratio", "loss"])]
for key in sorted(trl_keys):
    valid = history_df[key].dropna()
    if len(valid) > 0:
        steps = history_df.loc[valid.index, step_key]
        print(f"  {key}: {len(valid)} samples, steps {steps.min():.0f}-{steps.max():.0f}")
    else:
        print(f"  {key}: column exists but no data")
if not trl_keys:
    print("  No TRL metrics (entropy, kl, clip_ratio) found in history")

## Training Speed & Other Metrics

In [ ]:
def _find_key(
    history_df: pd.DataFrame,
    candidates: list[str],
) -> str | None:
    """Return the first candidate key that exists and has data in history_df."""
    for key in candidates:
        if key in history_df.columns and history_df[key].notna().any():
            return key
    return None


def plot_training_speed_metrics(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    time_key: str | None = None,
    figsize: tuple[int, int] = (14, 10),
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
    y_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure:
    """Plot training speed and throughput metrics (2x2 grid).

    Subplots: step time, samples/sec throughput, failure counts, training progress.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        time_key: Name of the time column (_timestamp or _runtime). If None, auto-detected.
        figsize: Figure size.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.
        y_percentile_range: Percentile range for y-axis limits (clips outliers).

    Returns:
        The matplotlib Figure object.
    """
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes_flat = axes.flatten()
    if time_key is None:
        time_key = "_runtime" if "_runtime" in history_df.columns else "_timestamp"
    # resolve keys: prefer new throughput/ keys, fall back to legacy HF keys
    train_sps = _find_key(history_df, ["train/throughput/samples_per_second", "train/samples_per_second"])
    eval_sps = _find_key(history_df, ["eval/throughput/samples_per_second", "eval/samples_per_second"])
    metrics_config: list[tuple[str, str, list[tuple[str | None, str]]]] = [
        ("Step Time", "seconds", [("train/step_time", "train"), ("eval/step_time", "eval")]),
        ("Samples / Second", "samples/s", [(train_sps, "train"), (eval_sps, "eval")]),
        (
            "Failure Counts",
            "count",
            [("train/failures/failure_count", "train"), ("eval/failures/failure_count", "eval")],
        ),
    ]
    # compute global x-axis limits from step-based metrics
    all_x_values: list[float] = []
    for _, _, key_splits in metrics_config:
        for key, _ in key_splits:
            if key and key in history_df.columns:
                valid_df = history_df[[step_key, key]].dropna()
                if len(valid_df) > 0:
                    all_x_values.extend(valid_df[step_key].values)
    x_limits = (
        _compute_percentile_x_limits(all_x_values, x_percentile_range) if x_percentile_range and all_x_values else None
    )
    # plot step-based metrics
    for ax_idx, (title, ylabel, key_splits) in enumerate(metrics_config):
        ax = axes_flat[ax_idx]
        plotted_any = False
        for key, split in key_splits:
            if not key or key not in history_df.columns:
                continue
            valid_df = history_df[[step_key, key]].dropna().sort_values(step_key).reset_index(drop=True)
            if len(valid_df) == 0:
                continue
            color = SPLIT_COLORS[split]
            _plot_metric_series(
                ax, valid_df[step_key].values, valid_df[key].values, color, f"{split} (n={len(valid_df)})"
            )
            plotted_any = True
        if not plotted_any:
            ax.text(0.5, 0.5, f"No {ylabel} metrics found", ha="center", va="center", transform=ax.transAxes)
        _configure_metric_ax(ax, step_key, title, x_limits, show_legend)
        ax.set_ylabel(ylabel)
        # clip y-axis to percentile range to handle outliers
        if plotted_any and y_percentile_range is not None:
            all_y = np.concatenate(
                [
                    history_df[[step_key, k]].dropna()[k].values
                    for k, _ in key_splits
                    if k and k in history_df.columns and history_df[k].notna().any()
                ]
            )
            if len(all_y) > 0:
                y_limits = _compute_percentile_x_limits(all_y, y_percentile_range, padding_fraction=0.1)
                if y_limits is not None:
                    ax.set_ylim(y_limits)
    # subplot 4: training progress (steps vs wall-clock time)
    ax = axes_flat[3]
    if time_key in history_df.columns:
        valid_df = history_df[[step_key, time_key]].dropna().sort_values(time_key)
        if len(valid_df) > 0:
            time_vals = valid_df[time_key].values.copy()
            step_vals = valid_df[step_key].values
            if time_key == "_timestamp":
                time_vals = time_vals - time_vals[0]
                x_label = "Time (seconds since start)"
            else:
                x_label = "Runtime (seconds)"
            if time_vals.max() > 3600:
                time_vals = time_vals / 3600
                x_label = x_label.replace("seconds", "hours")
            elif time_vals.max() > 60:
                time_vals = time_vals / 60
                x_label = x_label.replace("seconds", "minutes")
            ax.plot(time_vals, step_vals, color=SPLIT_COLORS["train"], linewidth=2)
            ax.scatter(
                time_vals[:: max(1, len(time_vals) // 50)],
                step_vals[:: max(1, len(step_vals) // 50)],
                color=SPLIT_COLORS["train"],
                alpha=0.5,
                s=20,
            )
            if len(time_vals) > 1 and time_vals[-1] > time_vals[0]:
                total_steps = step_vals[-1] - step_vals[0]
                total_time = time_vals[-1] - time_vals[0]
                unit = x_label.split("(")[-1].split(")")[0].split()[-1]
                ax.text(
                    0.02,
                    0.98,
                    f"Avg: {total_steps / total_time:.2f} steps/{unit}",
                    transform=ax.transAxes,
                    fontsize=9,
                    verticalalignment="top",
                    bbox={"boxstyle": "round", "facecolor": "wheat", "alpha": 0.5},
                )
            ax.set_xlabel(x_label)
            ax.set_ylabel(step_key)
            ax.set_title("Training Progress (Steps vs Wall-Clock)")
            ax.grid(True, alpha=0.3)
        else:
            ax.text(0.5, 0.5, "No time data available", ha="center", va="center", transform=ax.transAxes)
            ax.set_title("Training Progress")
    else:
        ax.text(0.5, 0.5, "No time key available", ha="center", va="center", transform=ax.transAxes)
        ax.set_title("Training Progress")
    return fig


fig = plot_training_speed_metrics(history_df, step_key=step_key, time_key=time_key)
plt.tight_layout()
plt.show()

## GPU Statistics

In [ ]:
def plot_gpu_stats(
    history_df: pd.DataFrame,
    step_key: str = "train/global_step",
    figsize: tuple[int, int] = (14, 10),
    show_legend: bool = True,
    x_percentile_range: tuple[float, float] | None = (1, 99),
) -> matplotlib.figure.Figure | None:
    """Plot GPU statistics over time (2x2 grid).

    Subplots: GPU utilization, VRAM usage, power draw, temperature.
    Shows mean values with std-based CI bands when available.

    Args:
        history_df: DataFrame with history data.
        step_key: Name of the step column (x-axis).
        figsize: Figure size.
        show_legend: Whether to show legends.
        x_percentile_range: Percentile range for x-axis limits.

    Returns:
        The matplotlib Figure, or None if no GPU data is available.
    """
    # check for any gpu columns
    gpu_cols = [c for c in history_df.columns if "/gpu/" in c]
    if not gpu_cols:
        return None
    fig, axes = plt.subplots(2, 2, figsize=figsize)
    axes_flat = axes.flatten()
    # define metrics: (title, ylabel, metric_base_name, y_range_hint)
    metrics_config = [
        ("GPU Utilization", "%", "utilization_gpu_percent", (0, 100)),
        ("VRAM Usage", "%", "vram_used_percent", (0, 100)),
        ("Power Draw", "watts", "power_watts", None),
        ("Temperature", "\u00b0C", "temperature_celsius", None),
    ]
    # compute global x-axis limits
    all_x_values: list[float] = []
    for _, _, base_name, _ in metrics_config:
        for prefix in ("train", "eval"):
            mean_key = f"{prefix}/gpu/{base_name}/mean"
            if mean_key in history_df.columns:
                valid_df = history_df[[step_key, mean_key]].dropna()
                if len(valid_df) > 0:
                    all_x_values.extend(valid_df[step_key].values)
    x_limits = (
        _compute_percentile_x_limits(all_x_values, x_percentile_range) if x_percentile_range and all_x_values else None
    )
    for ax_idx, (title, ylabel, base_name, y_hint) in enumerate(metrics_config):
        ax = axes_flat[ax_idx]
        plotted_any = False
        for split in ("train", "eval"):
            mean_key = f"{split}/gpu/{base_name}/mean"
            std_key = f"{split}/gpu/{base_name}/std"
            if mean_key not in history_df.columns:
                # fall back to pytorch_allocated_percent if vram_used_percent is missing
                if base_name == "vram_used_percent":
                    fallback = f"{split}/gpu/pytorch_allocated_percent/mean"
                    if fallback in history_df.columns:
                        mean_key = fallback
                        std_key = f"{split}/gpu/pytorch_allocated_percent/std"
                    else:
                        continue
                else:
                    continue
            cols = [step_key, mean_key]
            has_std = std_key in history_df.columns
            if has_std:
                cols.append(std_key)
            valid_df = history_df[cols].dropna(subset=[step_key, mean_key]).sort_values(step_key).reset_index(drop=True)
            if len(valid_df) == 0:
                continue
            _plot_metric_series(
                ax,
                valid_df[step_key].values,
                valid_df[mean_key].values,
                SPLIT_COLORS[split],
                f"{split} (n={len(valid_df)})",
                std_vals=valid_df[std_key].fillna(0).values if has_std else None,
                show_ci=True,
            )
            plotted_any = True
        if not plotted_any:
            ax.text(0.5, 0.5, f"No {base_name} data found", ha="center", va="center", transform=ax.transAxes)
        _configure_metric_ax(ax, step_key, title, x_limits, show_legend)
        ax.set_ylabel(ylabel)
        if y_hint is not None and plotted_any:
            ax.set_ylim(y_hint)
    return fig


# diagnostic: show what gpu columns are available
gpu_cols = sorted(c for c in history_df.columns if "/gpu/" in c)
if gpu_cols:
    print(f"Found {len(gpu_cols)} GPU columns in history_df:")
    for col in gpu_cols:
        n_valid = history_df[col].notna().sum()
        print(f"  {col}: {n_valid} non-null values")
    fig = plot_gpu_stats(history_df, step_key=step_key)
    if fig is not None:
        plt.tight_layout()
        plt.show()
    else:
        print("GPU columns found but no plottable data (check step_key alignment)")
else:
    print("No /gpu/ columns in history_df.")